# KOSIS ChromaDB 하이브리드 좌표 검색 → 2차 READY (Colab GPU)

1차 READY 이후의 ITEM/OBJ 좌표 후보를 ChromaDB dense + lexical + reranker 로 뽑고,
기존 KOSIS API 검증(`kosis_validate_mapping_candidates.py`)에 그대로 연결한다.

**원칙**: 임베딩/리랭커 점수는 후보 생성·순위에만 쓴다. READY 는 공식 메타 + KOSIS API 결과로만 확정한다.

**셀 순서**: 1 GPU 확인 → 2 저장소·의존성 → 3 Drive 마운트 → 4 입력 확인 →
5 Chroma 인덱스 생성 → 6 하이브리드 검색 → 7 API 검증 → 8 실제값 검증 →
9 A/B/C 평가 → 10 Drive 저장

In [ ]:
# 1. GPU 확인 (런타임 → 런타임 유형 변경 → GPU)
!nvidia-smi -L

In [ ]:
# 2. 저장소 + 의존성
!git clone https://github.com/rnwjdgus03/NLP_05-Team-Project-3.git repo
%cd repo
!git checkout codex/repro-baseline-20260727
!pip install -q -r requirements-ml.txt

In [ ]:
# 3. Drive 마운트 (기존 07_mapping_jinsung 결과 재사용)
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
os.environ['KOSIS_API_KEY'] = userdata.get('KOSIS_API_KEY') or ''  # 코드/CSV/로그에 키를 남기지 않는다
if not os.environ['KOSIS_API_KEY']:
    raise RuntimeError('Colab 보안 비밀에 KOSIS_API_KEY를 등록하세요.')

RUN = '/content/drive/MyDrive/NLP_05-Team-Project-3/runs/contextual_top50_context_v2_8x3/07_mapping_jinsung'
OUT = RUN + '/chroma_hybrid'
!mkdir -p {OUT}
!ls {RUN}

In [ ]:
# 4. 입력 확인 (평가 대상 measurement 를 여기서 고정한다)
import pandas as pd
ready = pd.read_csv(f'{RUN}/05_hcx_measurements_kosis_ready.csv')
meta = pd.read_csv(f'{RUN}/05_hcx_measurements_kosis_meta_index.csv')
cand = pd.read_csv(f'{RUN}/05_hcx_measurements_kosis_table_candidates.csv')
print('1차 READY measurement:', ready['claim_measurement_id'].nunique())
print('meta rows:', len(meta), '| table candidates:', len(cand))

In [ ]:
# 5. Chroma 좌표 인덱스 생성 (BGE-M3 임베딩을 직접 저장 → manifest 로 모델·차원 고정)
!python kosis_build_chroma_meta_index.py \
  --meta-index {RUN}/05_hcx_measurements_kosis_meta_index.csv \
  --persist-dir data/indexes/kosis_meta_chroma \
  --collection kosis_meta_coordinates \
  --embedding-model BAAI/bge-m3 \
  --prd-se-source {RUN}/05_hcx_measurements_kosis_table_candidates.csv \
  --device cuda \
  --reset
!cat data/indexes/kosis_meta_chroma/chroma_manifest.json

In [ ]:
# 6. 하이브리드 검색 (metadata filter → dense → lexical → RRF → reranker → Top-10)
!python kosis_chroma_hybrid_search.py \
  --claims {RUN}/05_hcx_measurements_kosis_ready.csv \
  --table-candidates {RUN}/05_hcx_measurements_kosis_table_candidates.csv \
  --persist-dir data/indexes/kosis_meta_chroma \
  --collection kosis_meta_coordinates \
  --output {OUT}/05_hcx_measurements_kosis_chroma_candidates.csv \
  --stats-output {OUT}/chroma_search_stats.csv \
  --dense-top-k 50 --lexical-top-k 50 --rerank-top-k 20 --final-top-k 10 \
  --reranker-model BAAI/bge-reranker-v2-m3 --device cuda

### 6-1. (선택) 실험 B — Chroma dense 만
`--no-reranker --lexical-top-k 0` 으로 dense 단독 후보를 만들어 A/B/C 비교에 쓴다.

In [ ]:
!python kosis_chroma_hybrid_search.py \
  --claims {RUN}/05_hcx_measurements_kosis_ready.csv \
  --table-candidates {RUN}/05_hcx_measurements_kosis_table_candidates.csv \
  --persist-dir data/indexes/kosis_meta_chroma \
  --collection kosis_meta_coordinates \
  --output {OUT}/05_hcx_measurements_kosis_dense_only_candidates.csv \
  --dense-top-k 50 --lexical-top-k 0 --rerank-top-k 20 --final-top-k 10 \
  --no-reranker --device cuda

In [ ]:
# 7. 기존 KOSIS API 검증에 연결 (READY만 자동 확정, PROVISIONAL은 수동 검토)
!python kosis_validate_mapping_candidates.py \
  --input {OUT}/05_hcx_measurements_kosis_chroma_candidates.csv \
  --meta-index {RUN}/05_hcx_measurements_kosis_meta_index.csv \
  --output {OUT}/05_hcx_measurements_kosis_chroma_validated.csv \
  --evaluate-all-ranks \
  --strict-seeded-coordinate \
  --item-top-k 1 --obj-top-k 1 --max-combinations 1 \
  --allow-provisional

In [ ]:
# 7-1. measurement 단위 진단 (885 후보행 → measurement 단위로 축약)
!python diagnose_validated_mappings.py \
  --validated {OUT}/05_hcx_measurements_kosis_chroma_validated.csv \
  --output {OUT}/diagnosis_chroma.csv

In [ ]:
# 8. 실제값 검증 (기사일 컬럼이 있어야 REVISION_VINTAGE_RISK 정책이 동작)
import pandas as pd
validated = pd.read_csv(f'{OUT}/05_hcx_measurements_kosis_chroma_validated.csv')
ready = pd.read_csv(f'{RUN}/05_hcx_measurements_kosis_ready.csv')
if 'date' not in validated.columns and 'date' in ready.columns:
    validated = validated.merge(ready[['claim_measurement_id', 'date']],
                                on='claim_measurement_id', how='left')
validated[validated['mapping_status'] == 'READY'].to_csv(
    f'{OUT}/verify_input_chroma.csv', index=False, encoding='utf-8-sig')
print('verify 대상:', (validated['mapping_status'] == 'READY').sum())

In [ ]:
!python kosis_verify_claim_values.py \
  --input {OUT}/verify_input_chroma.csv \
  --output {OUT}/05_hcx_measurements_kosis_chroma_verified.csv \
  --delay 0.12

In [ ]:
# 9. A/B/C 동일 표본 평가 (골드 좌표가 없으면 recall 은 gold_required 로 표시된다)
GOLD = ''  # 예: f'{RUN}/gold_coordinates.csv'
gold_arg = f'--gold {GOLD}' if GOLD else ''

!python evaluate_chroma_hybrid_mapping.py --label A_baseline \
  --measurements {RUN}/05_hcx_measurements_kosis_ready.csv \
  --candidates {RUN}/05_hcx_measurements_kosis_candidates_with_meta.csv \
  --validated {RUN}/05_hcx_measurements_kosis_validated_mappings.csv \
  --verified {RUN}/05_hcx_measurements_kosis_verified.csv {gold_arg} \
  --output {OUT}/eval_A.json

!python evaluate_chroma_hybrid_mapping.py --label B_chroma_dense \
  --measurements {RUN}/05_hcx_measurements_kosis_ready.csv \
  --candidates {OUT}/05_hcx_measurements_kosis_dense_only_candidates.csv {gold_arg} \
  --output {OUT}/eval_B.json

!python evaluate_chroma_hybrid_mapping.py --label C_chroma_hybrid \
  --measurements {RUN}/05_hcx_measurements_kosis_ready.csv \
  --candidates {OUT}/05_hcx_measurements_kosis_chroma_candidates.csv \
  --validated {OUT}/05_hcx_measurements_kosis_chroma_validated.csv \
  --verified {OUT}/05_hcx_measurements_kosis_chroma_verified.csv \
  --stats {OUT}/chroma_search_stats.csv {gold_arg} \
  --output {OUT}/eval_C.json

In [ ]:
# 11. 원인 분리 — A는 API-valid 인데 C 가 놓친 measurement (골드 없이 검색 품질 측정)
#
# 아이디어: A 가 KOSIS API 로 코드 일치까지 확인한 좌표는 '사실상 정답에 가까운 좌표'다.
#          그 좌표가 C 의 Top-10 후보 안에 들어 있었는지 보면, 실패가
#          '검색이 못 찾은 것'인지 '순위에서 밀린 것'인지 '표부터 틀린 것'인지 갈린다.
import pandas as pd

TRUE = {"true", "1", "y", "yes", "t"}
DEEP = 3          # 엄격 비교 깊이 (obj_l1~l3)


def load(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False)


def norm(v):
    v = str(v or "").strip()
    return "" if v.lower() in {"nan", "none"} else v


def api_ok(df, keys):
    if "response_code_valid" not in df.columns:
        raise KeyError("response_code_valid 없음 → 컬럼: " + ", ".join(list(df.columns)[:25]))
    flag = df["response_code_valid"].astype(str).str.strip().str.lower().isin(TRUE)
    return df[flag & df["claim_measurement_id"].isin(keys)]


def coord_cols(df):
    itm = "selected_itm_id" if "selected_itm_id" in df.columns else "itm_id"
    objs = []
    for i in range(1, 9):
        for col in (f"selected_obj_l{i}", f"obj_l{i}"):
            if col in df.columns:
                objs.append(col)
                break
    return itm, objs


def key_of(row, itm, objs, depth):
    return (norm(row.get("tbl_id")), norm(row.get(itm))) + tuple(
        norm(row.get(c)) for c in objs[:depth])


def rank_int(v):
    try:
        return int(float(v))
    except (TypeError, ValueError):
        return 999


A_val = load(f"{RUN}/05_hcx_measurements_kosis_validated_mappings.csv")
C_val = load(f"{OUT}/05_hcx_measurements_kosis_chroma_validated.csv")
C_cand = load(f"{OUT}/05_hcx_measurements_kosis_chroma_candidates.csv")
ready = load(f"{RUN}/05_hcx_measurements_kosis_ready.csv")
try:
    stats = load(f"{OUT}/chroma_search_stats.csv").drop_duplicates(
        "claim_measurement_id").set_index("claim_measurement_id")
except Exception as exc:
    print("stats 로드 실패:", exc)
    stats = None

KEYS = set(ready["claim_measurement_id"])
text_of = dict(zip(ready["claim_measurement_id"],
                   ready["claim_text"] if "claim_text" in ready.columns else [""] * len(ready)))

A_ok, C_ok = api_ok(A_val, KEYS), api_ok(C_val, KEYS)
A_set, C_set = set(A_ok["claim_measurement_id"]), set(C_ok["claim_measurement_id"])

print(f"평가 대상 {len(KEYS)} | A api-valid {len(A_set)} | C api-valid {len(C_set)}")
print(f"둘 다 {len(A_set & C_set)} | A만 {len(A_set - C_set)} | "
      f"C만 {len(C_set - A_set)} | 둘 다 실패 {len(KEYS - A_set - C_set)}")

itmA, objsA = coord_cols(A_val)
itmC, objsC = coord_cols(C_cand)
print("좌표 컬럼  A:", itmA, objsA[:DEEP], "| C:", itmC, objsC[:DEEP])

C_by_m = dict(tuple(C_cand.groupby("claim_measurement_id")))

rows = []
for mid in sorted(A_set):
    a = A_ok[A_ok["claim_measurement_id"] == mid]
    want_loose = {key_of(r, itmA, objsA, 1) for _, r in a.iterrows()}
    want_deep = {key_of(r, itmA, objsA, DEEP) for _, r in a.iterrows()}
    want_tbl = {k[0] for k in want_loose}
    g = C_by_m.get(mid)

    if g is None or g.empty:
        cls, hit_rank, deep_hit = "NO_CANDIDATE", "", False
    else:
        got_loose, got_deep, got_tbl = {}, set(), set()
        for _, r in g.iterrows():
            got_loose.setdefault(key_of(r, itmC, objsC, 1), rank_int(r.get("candidate_rank")))
            got_deep.add(key_of(r, itmC, objsC, DEEP))
            got_tbl.add(norm(r.get("tbl_id")))
        hits = [got_loose[k] for k in want_loose if k in got_loose]
        deep_hit = bool(want_deep & got_deep)
        if hits:
            cls, hit_rank = "COORD_IN_TOPK", min(hits)
        elif want_tbl & got_tbl:
            cls, hit_rank = "TABLE_OK_COORD_MISS", ""
        else:
            cls, hit_rank = "TABLE_MISS", ""

    st = stats.loc[mid].to_dict() if stats is not None and mid in stats.index else {}
    rows.append({
        "claim_measurement_id": mid,
        "c_api_valid": mid in C_set,
        "failure_class": cls,
        "coord_hit_rank": hit_rank,
        "deep_match": deep_hit,
        "a_tbl_id": " | ".join(sorted(want_tbl)),
        "a_coordinate": " | ".join(sorted("/".join(k[1:]) for k in want_deep))[:200],
        "c_candidate_rows": 0 if g is None else len(g),
        "dense_count": st.get("dense_count", ""),
        "lexical_count": st.get("lexical_count", ""),
        "claim_text": str(text_of.get(mid, ""))[:120],
    })

diag = pd.DataFrame(rows)
diag.to_csv(f"{OUT}/why_chroma_missed.csv", index=False, encoding="utf-8-sig")

print(f"\n=== A 의 API-valid 좌표가 C Top-10 안에 있었나 (분모 {len(diag)}) ===")
print(diag["failure_class"].value_counts().to_string())
loose = (diag["failure_class"] == "COORD_IN_TOPK").mean()
deep = diag["deep_match"].mean()
print(f"\n좌표 재현율 Top-10  느슨(tbl+itm+obj_l1) = {loose:.1%} | 엄격(obj_l3까지) = {deep:.1%}")
print("  ↑ 골드 없이 측정 가능한 검색 품질. 낮으면 '검색이 못 찾은 것'.")

top = diag[diag["failure_class"] == "COORD_IN_TOPK"]["coord_hit_rank"]
if len(top):
    print("\n맞춘 좌표의 rank 분포:")
    print(top.value_counts().sort_index().to_string())

miss = diag[~diag["c_api_valid"]]
print(f"\n=== A만 성공하고 C가 놓친 {len(miss)}건의 실패 유형 ===")
print(miss["failure_class"].value_counts().to_string())
print()
print(miss[["claim_measurement_id", "failure_class", "coord_hit_rank",
            "a_tbl_id", "claim_text"]].head(25).to_string(index=False))

no_cand = KEYS - set(C_cand["claim_measurement_id"])
print(f"\n=== C 가 후보를 아예 못 만든 measurement {len(no_cand)}건 ===")
for mid in sorted(no_cand):
    print(" ", mid, "|", str(text_of.get(mid, ""))[:90])
if no_cand and stats is not None:
    have = [m for m in sorted(no_cand) if m in stats.index]
    if have:
        print("\n해당 건 검색 통계 (필터가 후보를 다 걷어냈는지 확인):")
        print(stats.loc[have].to_string())

print(f"\n저장: {OUT}/why_chroma_missed.csv")


In [ ]:
# 10. 결과 저장 (Chroma 인덱스는 용량이 크므로 Git 에 커밋하지 않는다)
!cp -r data/indexes/kosis_meta_chroma {OUT}/kosis_meta_chroma_index
!ls -la {OUT}

## 해석 주의
- 후보행 수(measurement × Top-K)를 measurement 실패 건수로 읽지 말 것.
- 골드 좌표(`gold_tbl_id` / `gold_itm_id` / `gold_obj_l1`)가 없으면 recall 은 계산하지 않는다.
- READY 증가만으로 개선을 주장하지 말고, 동일 표본 A/B/C 결과와 수동 검수 Precision 으로 판단한다.